# 07 — Index E (Global-ESG)

**Objetivo.** Predecir Index_E con el enfoque más sencillo que mejore el baseline.

**Pistas del enunciado.** Global-ESG — mezcla de empresas sostenibles/globales. Sin pista específica de datos auxiliares.

**Nivel de esfuerzo.** MEDIO. Si el baseline ya es competitivo, no forzar arquitecturas complejas. Un LSTM modesto es el primer intento.

**Estrategia.** Verificar primero si baseline drift/flat ya da un RMSE razonable. Si no, LSTM estándar en log-ret mode. Sin ensemble de seeds salvo que la diferencia en RMSE total justifique el tiempo.

**Qué esperamos.** Comportamiento mixto — el LSTM probablemente mejora algo sobre el baseline, pero E no domina el RMSE total. No sobreoptimizar aquí.

**Qué NO hace.** No usa datos auxiliares. No hace ensemble de seeds por defecto.

**Inputs.** `data/train_indices.csv`, `results/baselines.json`

**Output esperado.** `models/{OWNER}_Index_E.keras` (si NN gana), `results/index_E.json`

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

OWNER = "oscar"   # <-- cada miembro pone su nombre aquí
IDX   = "Index_E"

import json
import numpy as np
import matplotlib.pyplot as plt

from utils import (
    load_hackathon_data, make_window_dataset, make_temporal_split,
    build_model, train_model,
    backtest_autoregressive, baseline_flat, baseline_drift,
    plot_history, plot_rollout,
    DATA_DIR, VAL_DAYS, V_IN_SHARED, RANDOM_SEED
)

os.makedirs('models',  exist_ok=True)
os.makedirs('results', exist_ok=True)
MODEL_PATH = f'models/{OWNER}_{IDX}.keras'

## 1. Carga + diagnóstico + referencia baselines

In [ ]:
data  = load_hackathon_data(DATA_DIR)
idx   = data['train_indices']
serie = idx[IDX].dropna().values

plt.figure(figsize=(13, 3))
plt.plot(serie, lw=0.8)
plt.title(f'{IDX} — precios brutos')
plt.tight_layout()
plt.show()

with open('results/baselines.json') as f:
    baselines = json.load(f)
print(f'Baselines para {IDX}:')
for bl, rmse in baselines[IDX].items():
    print(f'  {bl:<15} RMSE = {rmse:,.0f}')

best_bl_name = min(baselines[IDX], key=baselines[IDX].get)
best_bl_rmse = baselines[IDX][best_bl_name]

## 2. Visualización rápida del mejor baseline

Si el rollout ya parece razonable, el baseline puede bastar para E.

In [ ]:
serie_train, _ = make_temporal_split(serie, val_days=VAL_DAYS, v_in=V_IN_SHARED)

preds_bl = baseline_flat(serie_train, VAL_DAYS) if best_bl_name == 'flat' \
           else baseline_drift(serie_train, VAL_DAYS)

plot_rollout(serie, preds_bl, index_name=f'{IDX} — {best_bl_name}', val_days=VAL_DAYS)

## 3. LSTM estándar — solo si el baseline no basta

In [ ]:
import tensorflow as tf
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

X, y = make_window_dataset(serie_train, V_IN_SHARED, use_log_rets=True)
cut  = int(len(X) * 0.8)
X_tr, y_tr = X[:cut], y[:cut]
X_v,  y_v  = X[cut:], y[cut:]

model = build_model('lstm', V_IN_SHARED, n_features=1)
hist  = train_model(model, X_tr, y_tr, X_v, y_v, epochs=300)
plot_history(hist, title=f'{IDX} — LSTM')

In [ ]:
predict_fn = lambda x: model.predict(x, verbose=0).ravel()[0]
bt = backtest_autoregressive(predict_fn, serie, log_ret_mode=True)
print(f'LSTM RMSE backtest = {bt["rmse"]:,.0f}  |  dir_acc = {bt["dir_accuracy"]:.2%}')
mejora = (best_bl_rmse - bt['rmse']) / best_bl_rmse * 100
print(f'Mejora vs baseline: {mejora:+.1f}%')
plot_rollout(serie, bt['preds'], index_name=f'{IDX} — LSTM', val_days=VAL_DAYS)

## 4. Decisión final y guardado

In [ ]:
_tipo_map = {'flat': 'baseline_flat', 'drift': 'baseline_drift', 'random_walk': 'baseline_rw'}

if bt['rmse'] < best_bl_rmse:
    model.save(MODEL_PATH)
    approach   = 'nn'
    final_rmse = bt['rmse']
    final_path = MODEL_PATH
    notes      = 'LSTM estándar, 300 epochs'
else:
    approach   = _tipo_map[best_bl_name]
    final_rmse = best_bl_rmse
    final_path = None
    notes      = f'LSTM no mejora baseline — usando {best_bl_name}'

info = {
    'index':              IDX,
    'owner':              OWNER,
    'approach_type':      approach,
    'strategy':           approach,
    'rmse_backtest_252d': final_rmse,
    'model_path':         final_path,
    'log_ret_mode':       True if approach == 'nn' else False,
    'v_in':               V_IN_SHARED if approach == 'nn' else None,
    'n_features':         1,
    'aux_source':         None,
    'aux_test_source':    None,
    'aux_columns':        None,
    'ghost_source_index': None,
    'ghost_lag':          None,
    'notes':              notes
}

with open('results/index_E.json', 'w') as f:
    json.dump(info, f, indent=2)

print('Guardado: results/index_E.json')
print(json.dumps(info, indent=2))